In [19]:
%pip install opencv-python numpy imutils matplotlib
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import threading
import time

Note: you may need to restart the kernel to use updated packages.


In [20]:
# takes 4 points and orders them in tl, tr, br, bl order by understanding which are the left coordinates and which are the right coordinates
# then it uses the y coordinate to distinguish top and bottom points
# returns the ordered points as a numpy array of float32
def order_points(pts):
    pts = pts[np.argsort(pts[:,0])]
    left = pts[:2]
    right = pts[2:]

    tl, bl = left[np.argsort(left[:,1])]
    tr, br = right[np.argsort(right[:,1])]
    return np.array([tl, tr, br, bl], dtype=np.float32)

# takes the 4 ordered points and the image and defines the destination points for a perspective transform
# then it computes the perspective transform matrix and applies it to the image to get a top-down view of the area defined by the points
# returns the warped image
# 0,0 is top-left, size,0 is top-right, size,size is bottom-right, 0,size is bottom-left
def warp_grid(img, pts, size=900):
    pts = order_points(pts)
    dst = np.array([[0,0],[size,0],[size,size],[0,size]], dtype=np.float32)
    M = cv2.getPerspectiveTransform(pts, dst) # the trasnformation matrix that allows us to understand how we are supposed to warp the image
    return cv2.warpPerspective(img, M, (size,size)) # returns the warped image

# this function takes a greyscale image then takes the border regions and calculates their mean brightness
# if the mean brightness is below a certain threshold (127), it considers the image to be inverted (dark background with light foreground)
# returns True if inverted, False otherwise
def is_inverted(gray_img):
    h, w = gray_img.shape
    
    # Sample border regions (top, bottom, left, right edges)
    border_width = int(min(h, w) * 0.1)  # 10% of smaller dimension
    
    top_border = gray_img[:border_width, :]
    bottom_border = gray_img[-border_width:, :]
    left_border = gray_img[:, :border_width]
    right_border = gray_img[:, -border_width:]
    
    # Calculate mean brightness of borders
    border_brightness = np.mean([
        np.mean(top_border),
        np.mean(bottom_border),
        np.mean(left_border),
        np.mean(right_border)
    ])
    
    # If border is dark, image is inverted
    return border_brightness < 127

In [ ]:
def detect_grid_with_contour(img):
    h, w = img.shape[:2]
    scale = 900 / max(h, w)
    img_resized = cv2.resize(img, None, fx=scale, fy=scale)
    
    # Convert to grayscale
    gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
    
    # Check for inverted colors
    if is_inverted(gray):
        print(f"  [Contour] Inverted image detected, fixing...")
        gray = 255 - gray
    
    # Preprocess
    mean_brightness = np.mean(gray)
    
    # Apply CLAHE so we can improve the contrast of each image
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    gray = clahe.apply(gray)
    
    # Blur to reduce noise so we can reduce the noise the image has
    blur = cv2.GaussianBlur(gray, (5,5), 0)
    
    # then we create different thresholded images to try and find the grid
    thresh_methods = []
    
    # Method 1: Adaptive threshold (this is better for fine details)
    thresh1 = cv2.adaptiveThreshold(blur, 255,
                                    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                    cv2.THRESH_BINARY_INV, 11, 2)
    thresh_methods.append(thresh1)
    
    # Method 2: Larger block size (this is better for images with different lighting)
    thresh2 = cv2.adaptiveThreshold(blur, 255,
                                    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                    cv2.THRESH_BINARY_INV, 21, 2)
    thresh_methods.append(thresh2)
    
    # now we find all contours and sort them by area and then select the one with the largest area which is likely to be the grid
    for binary in thresh_methods:
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        if not contours:
            continue
        
        # Sort by area
        contours = sorted(contours, key=cv2.contourArea, reverse=True)
        
        # Try to find 4-corner contour
        epsilon_values = [0.02, 0.03, 0.04, 0.05, 0.01]
        
        for c in contours[:10]: # we check the top 10 largest contours
            peri = cv2.arcLength(c, True)
            
            for eps in epsilon_values:
                approx = cv2.approxPolyDP(c, eps * peri, True)
                
                if len(approx) == 4: # if we find a contour with 4 corners
                    x, y, w, h = cv2.boundingRect(approx)
                    aspect_ratio = w / float(h) if h > 0 else 0
                    area = cv2.contourArea(approx)
                    
                    if 0.7 < aspect_ratio < 1.4 and area > 50000: # we check if the aspect ratio is between 0.7 and 1.4 and the area is larger than 50000 pixels
                        corners = approx.reshape(4, 2).astype(np.float32)
                        print(f"  [Contour] Found grid with area {area:.0f}") # if all of that happens we found a grid
                        return corners, img_resized
    
    # Fallback: use bounding box of largest contour
    if contours: # if we don't find a 4-corner contour, we use the bounding box of the largest contour
        largest = contours[0]
        area = cv2.contourArea(largest)
        if area > 50000:
            x, y, w, h = cv2.boundingRect(largest)
            aspect = w / float(h)
            if 0.5 < aspect < 2.0:
                corners = np.array([[x, y], [x+w, y], [x+w, y+h], [x, y+h]], dtype=np.float32)
                print(f"  [Contour] Using bounding box with area {area:.0f}")
                return corners, img_resized
    
    return None, None # we return nothing if we don't find anything


# Create output directory
output_folder = "output_preprocess"
os.makedirs(output_folder, exist_ok=True)

# First of all, we convert the image to gray scale then the first check we make is if the image has salt and  paper noise. if so, we take it to a pipeline where we solve the noise. 
# Then we check the brightness of the image, and if the image is inverted. if the image was already dark we check its brightness and based on the brightness we apply something certain. 
# Then we apply normal processing for normal images then we start with edge detection, based on std brightness, we detect edges using a certain method. 
# Then we apply hough transformation to find the lines then we filter the lines into horizontal and vertical lines and start merging the lines. 
# Then using the lines, we try to find the outermost valid rectangle. 
def detect_grid_with_hough(img):
    """
    Detect sudoku grid using Hough Line Transform with proper line merging.
    INCLUDES: Full brightening pipeline + Morphological gradient for low contrast
    """
    steps = {}

    # Step 1: Resize and convert to grayscale
    h, w = img.shape[:2]
    scale = 900 / max(h, w)
    img_resized = cv2.resize(img, None, fx=scale, fy=scale) # we resize the image to 900 pixels
    steps['original'] = img_resized.copy() # we add the image to the array steps

    gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY) # we convert the image to grayscale
    steps['grayscale'] = gray.copy() # then add the image to the array steps but as the grayscale version
    
    # NOISE DETECTION: Check if image has heavy grain/salt-and-pepper noise just like what image 10 has
    # Strategy: Check edge density - noisy images have too many edges
    test_blur = cv2.GaussianBlur(gray, (5, 5), 0)
    test_edges = cv2.Canny(test_blur, 50, 150, apertureSize=3)
    edge_density = cv2.countNonZero(test_edges) / (test_edges.shape[0] * test_edges.shape[1])
    
    is_noisy = edge_density > 0.3  # More than 30% edge pixels = noisy
    
    if is_noisy:
        print(f"  [Hough] NOISY image detected! Edge density: {edge_density:.2%}")
        print(f"  [Hough] Applying aggressive denoising...")
        
        # Strategy: Multiple passes of denoising to remove grain
        # Pass 1: Median blur 
        gray = cv2.medianBlur(gray, 5)
        
        # Pass 2: Non-Local Means Denoising 
        gray = cv2.fastNlMeansDenoising(gray, h=15, templateWindowSize=7, searchWindowSize=21)
        
        # Pass 3: Another median blur to clean up
        gray = cv2.medianBlur(gray, 3)
        
        # Pass 4: Bilateral filter to smooth while preserving strong edges
        gray = cv2.bilateralFilter(gray, 9, 75, 75)
        
        print(f"  [Hough] After denoising: {np.mean(gray):.1f}")
        steps['grayscale'] = gray.copy()
    
    # Check brightness BEFORE inversion
    original_brightness = np.mean(gray)
    was_originally_dark = original_brightness < 60 # for images 7 and 8
    
    # Check for inverted images
    if is_inverted(gray):
        print(f"  [Hough] Inverted image detected! Original brightness: {original_brightness:.1f}")
        gray = 255 - gray
        steps['grayscale'] = gray.copy()
    
    mean_brightness = np.mean(gray)
    std_brightness = np.std(gray)
    
    print(f"  [Hough] Mean brightness: {mean_brightness:.1f}, Std: {std_brightness:.1f}")
    
    # For originally dark images that got inverted, check the current brightness
    if was_originally_dark:
        print(f"  [Hough] Image was originally dark, applying preprocessing...")
        
        # If after inversion brightness is high (>200), we're looking at low-contrast inverted image
        # The ACTUAL dark pixels are now (255 - gray), so work with those
        if mean_brightness > 200:
            print(f"  [Hough] Low contrast after inversion, inverting back...")
            gray = 255 - gray
            mean_brightness = np.mean(gray)
            print(f"  [Hough] After inverting back: {mean_brightness:.1f}")
    
    # SPECIAL BRANCH: Extremely dark image (< 40) # image 8
    if mean_brightness < 40:
        print(f"  [Hough] Extremely dark! Applying full brightening pipeline")
        
        # Step 0: DENOISE FIRST
        gray = cv2.bilateralFilter(gray, 9, 75, 75)
        gray = cv2.medianBlur(gray, 5)
        print(f"  [Hough] After denoising: {np.mean(gray):.1f}")
        
        # Step 1: Brightness gain 4.0x
        gain = 4.0
        gray = np.clip(gray.astype(np.float32) * gain, 0, 255).astype(np.uint8)
        print(f"  [Hough] After gain: {np.mean(gray):.1f}")
        
        # Step 2: Gamma correction
        gamma = 0.35
        gray = np.power(gray / 255.0, gamma) * 255.0
        gray = gray.astype(np.uint8)
        print(f"  [Hough] After gamma: {np.mean(gray):.1f}")
        
        # Step 3: CLAHE
        clahe = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8, 8))
        gray = clahe.apply(gray)
        print(f"  [Hough] After CLAHE: {np.mean(gray):.1f}")

        # Step 4: Check if STILL low contrast after all brightening
        if np.std(gray) < 10:
            print(f"  [Hough] STILL low contrast (std={np.std(gray):.1f})! Applying adaptive histogram equalization...")
            # Use adaptive histogram equalization for very stubborn low-contrast images
            gray = cv2.equalizeHist(gray)
            print(f"  [Hough] After equalizeHist: mean={np.mean(gray):.1f}, std={np.std(gray):.1f}")

        steps['grayscale'] = gray.copy()
    
    # Very dark image (but not extremely dark)
    elif mean_brightness < 60:
        print(f"  [Hough] Very dark image, applying gamma correction...")
        gamma = 2.0
        gray = np.power(gray / 255.0, 1.0 / gamma) * 255.0
        gray = gray.astype(np.uint8)
        print(f"  [Hough] After gamma: {np.mean(gray):.1f}")
        steps['grayscale'] = gray.copy()
    
    # Very light/faded image
    elif mean_brightness > 180 and std_brightness < 30:
        print(f"  [Hough] Very light/faded image, enhancing faint lines...")
        
        # Strategy: Use morphological operations to enhance faint lines
        # Step 1: Invert so grid lines become dark
        gray_inv = 255 - gray
        
        # Step 2: Apply morphological closing to connect faint line segments
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
        gray_inv = cv2.morphologyEx(gray_inv, cv2.MORPH_CLOSE, kernel)
        
        # Step 3: Enhance contrast with CLAHE on inverted image
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        gray_inv = clahe.apply(gray_inv)
        
        # Step 4: Invert back
        gray = 255 - gray_inv
        
        # Step 5: Apply adaptive thresholding to make lines more visible
        # Use a large block size to preserve grid structure
        thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                       cv2.THRESH_BINARY, 21, 5)
        
        # Step 6: Invert threshold result (grid lines should be dark)
        gray = 255 - thresh
        
        # Step 7: Light blur to smooth
        gray = cv2.GaussianBlur(gray, (3, 3), 0)
        
        print(f"  [Hough] After enhancement: mean={np.mean(gray):.1f}, std={np.std(gray):.1f}")
        steps['grayscale'] = gray.copy()
    
    # Standard processing for normal brightness (this is for images that are normal)
    if not was_originally_dark or (mean_brightness >= 60 and mean_brightness <= 180):
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
        gray = clahe.apply(gray)
    
    # Blur to reduce noise
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # Step 4: Adaptive edge detection
    std_brightness = np.std(blurred)
    
    if std_brightness < 15:
        # Very low contrast - use morphological gradient
        print(f"  [Hough] Low contrast (std={std_brightness:.1f}), using Morphological Gradient")
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
        gradient = cv2.morphologyEx(blurred, cv2.MORPH_GRADIENT, kernel)
        _, edges = cv2.threshold(gradient, 10, 255, cv2.THRESH_BINARY)
    elif std_brightness < 30:
        print(f"  [Hough] Medium contrast (std={std_brightness:.1f}), using Canny (30,100)")
        edges = cv2.Canny(blurred, 30, 100, apertureSize=3)
    else:
        print(f"  [Hough] Good contrast (std={std_brightness:.1f}), using Canny (50,150)")
        edges = cv2.Canny(blurred, 50, 150, apertureSize=3)
    
    print(f"  [Hough] Detected {cv2.countNonZero(edges)} edge pixels")
    steps['edges'] = edges.copy()

    # Step 5: Hough Line Detection
    lines = cv2.HoughLines(edges, 1, np.pi/180, threshold=150)

    if lines is None:
        print(f"  [Hough] No lines detected!")
        return steps, None
    
    # Visualize all detected lines
    hough_lines_img = img_resized.copy()
    for rho, theta in lines[:, 0]:
        a = np.cos(theta)
        b = np.sin(theta)
        x0 = a * rho
        y0 = b * rho
        x1 = int(x0 + 2000 * (-b))
        y1 = int(y0 + 2000 * (a))
        x2 = int(x0 - 2000 * (-b))
        y2 = int(y0 - 2000 * (a))

        if abs(theta - np.pi/2) < 0.2:
            color = (0, 0, 255)
        else:
            color = (255, 0, 0)

        cv2.line(hough_lines_img, (x1, y1), (x2, y2), color, 1)

    steps['hough_lines'] = hough_lines_img.copy()

    # Step 6: Filter and separate horizontal/vertical lines
    horizontal_lines = []
    vertical_lines = []

    for rho, theta in lines[:, 0]:
        if abs(theta) < 0.2 or abs(theta - np.pi) < 0.2:
            vertical_lines.append((rho, theta))
        elif abs(theta - np.pi/2) < 0.2:
            horizontal_lines.append((rho, theta))

    print(f"  [Hough] Raw lines: {len(horizontal_lines)} horizontal, {len(vertical_lines)} vertical")

    # Step 7: MERGE similar lines
    # Use larger threshold if image was rotated (image is larger after rotation)
    merge_threshold = 60 if hasattr(detect_grid_with_hough, '_was_rotated') else 30
    
    def merge_lines(lines, threshold=merge_threshold):
        if not lines:
            return []
        lines = sorted(lines, key=lambda x: x[0])
        merged = []
        current_group = [lines[0]]
        for i in range(1, len(lines)):
            if abs(lines[i][0] - current_group[-1][0]) < threshold:
                current_group.append(lines[i])
            else:
                avg_rho = np.mean([l[0] for l in current_group])
                avg_theta = np.mean([l[1] for l in current_group])
                merged.append((avg_rho, avg_theta))
                current_group = [lines[i]]
        if current_group:
            avg_rho = np.mean([l[0] for l in current_group])
            avg_theta = np.mean([l[1] for l in current_group])
            merged.append((avg_rho, avg_theta))
        return merged

    horizontal_lines = merge_lines(horizontal_lines)
    vertical_lines = merge_lines(vertical_lines)

    print(f"  [Hough] After merging: {len(horizontal_lines)} horizontal, {len(vertical_lines)} vertical")

    if len(horizontal_lines) < 2 or len(vertical_lines) < 2:
        print(f"  [Hough] Not enough lines to form grid!")
        return steps, None

    # Visualize filtered lines
    filtered_lines_img = img_resized.copy()

    for rho, theta in horizontal_lines:
        a = np.cos(theta)
        b = np.sin(theta)
        x0 = a * rho
        y0 = b * rho
        x1 = int(x0 + 2000 * (-b))
        y1 = int(y0 + 2000 * (a))
        x2 = int(x0 - 2000 * (-b))
        y2 = int(y0 - 2000 * (a))
        cv2.line(filtered_lines_img, (x1, y1), (x2, y2), (255, 0, 0), 2)

    for rho, theta in vertical_lines:
        a = np.cos(theta)
        b = np.sin(theta)
        x0 = a * rho
        y0 = b * rho
        x1 = int(x0 + 2000 * (-b))
        y1 = int(y0 + 2000 * (a))
        x2 = int(x0 - 2000 * (-b))
        y2 = int(y0 - 2000 * (a))
        cv2.line(filtered_lines_img, (x1, y1), (x2, y2), (0, 0, 255), 2)

    steps['filtered_lines'] = filtered_lines_img.copy()

    # Step 8: Find the outermost valid rectangle (prioritize boundary lines)
    def line_intersection(line1, line2):
        rho1, theta1 = line1
        rho2, theta2 = line2
        A = np.array([
            [np.cos(theta1), np.sin(theta1)],
            [np.cos(theta2), np.sin(theta2)]
        ])
        b = np.array([rho1, rho2])
        try:
            intersection = np.linalg.solve(A, b)
            return intersection
        except np.linalg.LinAlgError:
            return None

    def compute_rectangle_area(h1, h2, v1, v2):
        tl = line_intersection(h1, v1)
        tr = line_intersection(h1, v2)
        bl = line_intersection(h2, v1)
        br = line_intersection(h2, v2)
        if any(pt is None for pt in [tl, tr, bl, br]):
            return 0, None
        width = np.linalg.norm(tr - tl)
        height = np.linalg.norm(bl - tl)
        area = width * height
        return area, np.array([tl, tr, br, bl], dtype=np.float32)

    # BALANCED APPROACH: Search for largest valid rectangle with bounds checking
    horizontal_lines_sorted = sorted(horizontal_lines, key=lambda x: x[0])
    vertical_lines_sorted = sorted(vertical_lines, key=lambda x: x[0])
    
    best_area = 0
    best_corners = None
    best_lines = None

    img_h, img_w = img_resized.shape[:2]
    min_area = (img_w * img_h) * 0.1
    
    # This should capture the full grid without cutting off rows/columns
    if len(horizontal_lines_sorted) >= 2 and len(vertical_lines_sorted) >= 2:
        h_extreme_top = horizontal_lines_sorted[0]   # Smallest rho = topmost
        h_extreme_bot = horizontal_lines_sorted[-1]  # Largest rho = bottommost
        v_extreme_1 = vertical_lines_sorted[0]       # Most negative/leftmost-ish
        v_extreme_2 = vertical_lines_sorted[-1]      # Least negative/rightmost-ish
        
        extreme_area, extreme_corners = compute_rectangle_area(h_extreme_top, h_extreme_bot, v_extreme_1, v_extreme_2)
                
        if extreme_corners is not None and extreme_area > min_area:
            # Check corners are within bounds (with generous margin)
            margin = 50 
            valid_extreme = True
            for pt in extreme_corners:
                if pt[0] < -margin or pt[0] >= img_w + margin or pt[1] < -margin or pt[1] >= img_h + margin:
                    # print(f"  [Hough] DEBUG: Corner {pt} OUTSIDE bounds (margin={margin})")
                    valid_extreme = False
                    break
            
            if valid_extreme:
                for j in range(len(extreme_corners)):
                    extreme_corners[j][0] = max(0, min(img_w - 1, extreme_corners[j][0]))
                    extreme_corners[j][1] = max(0, min(img_h - 1, extreme_corners[j][1]))
                
                # Sort by y first (top vs bottom), then by x within each pair
                corners_list = [tuple(c) for c in extreme_corners]
                # Find top two (smallest y) and bottom two (largest y)
                sorted_by_y = sorted(corners_list, key=lambda c: c[1])
                top_two = sorted_by_y[:2]
                bot_two = sorted_by_y[2:]
                # Within top, leftmost is TL, rightmost is TR
                top_left = min(top_two, key=lambda c: c[0])
                top_right = max(top_two, key=lambda c: c[0])
                # Within bottom, leftmost is BL, rightmost is BR
                bot_left = min(bot_two, key=lambda c: c[0])
                bot_right = max(bot_two, key=lambda c: c[0])
                
                extreme_corners = np.array([top_left, top_right, bot_right, bot_left], dtype=np.float32)
                print(f"  [Hough] Reordered corners: TL={top_left}, TR={top_right}, BR={bot_right}, BL={bot_left}")
                
                best_area = extreme_area
                best_corners = extreme_corners
                best_lines = (h_extreme_top, h_extreme_bot, v_extreme_1, v_extreme_2)
                best_score = float('inf')
                print(f"  [Hough] Using extreme boundary lines")
            else:
                print(f"  [Hough] Corners outside bounds, falling back")
        else:
            print(f"  [Hough] No extreme area or corners, falling back")
    
    # This prioritizes rectangles that use extreme/boundary lines
    best_score = 0
    
    for i in range(len(horizontal_lines_sorted)):
        for j in range(i+1, len(horizontal_lines_sorted)):
            for k in range(len(vertical_lines_sorted)):
                for l in range(k+1, len(vertical_lines_sorted)):
                    h1, h2 = horizontal_lines_sorted[i], horizontal_lines_sorted[j]
                    v1, v2 = vertical_lines_sorted[k], vertical_lines_sorted[l]
                    area, corners = compute_rectangle_area(h1, h2, v1, v2)
                    
                    if corners is not None and area > min_area:
                        valid = True
                        for pt in corners:
                            if pt[0] < 0 or pt[0] >= img_w or pt[1] < 0 or pt[1] >= img_h:
                                valid = False
                                break
                        
                        if valid:
                            # Calculate line separation (span)
                            h_span = abs(h2[0] - h1[0])
                            v_span = abs(v2[0] - v1[0])
                            
                            # Score = area * (1 + normalized_span_bonus)
                            # Reduced span bonus from 2.0 to 1.0 for better balance
                            span_bonus = (h_span + v_span) / (img_h + img_w)
                            score = area * (1 + 1.0 * span_bonus)
                            
                            if score > best_score:
                                best_score = score
                                best_area = area
                                best_corners = corners
                                best_lines = (h1, h2, v1, v2)

    if best_corners is None:
        print(f"  [Hough] Could not find valid grid rectangle!")
        return steps, None

    print(f"  [Hough] Best rectangle area: {best_area:.0f} pixels")
    if best_lines:
        h1, h2, v1, v2 = best_lines
        print(f"  [Hough] Selected H lines: rho={h1[0]:.1f} (top), rho={h2[0]:.1f} (bottom)")
        print(f"  [Hough] Selected V lines: rho={v1[0]:.1f} (left), rho={v2[0]:.1f} (right)")

    # Visualize corners
    corners_img = img_resized.copy()

    for line in best_lines[:2]:
        rho, theta = line
        a = np.cos(theta)
        b = np.sin(theta)
        x0 = a * rho
        y0 = b * rho
        x1 = int(x0 + 2000 * (-b))
        y1 = int(y0 + 2000 * (a))
        x2 = int(x0 - 2000 * (-b))
        y2 = int(y0 - 2000 * (a))
        cv2.line(corners_img, (x1, y1), (x2, y2), (255, 0, 0), 2)

    for line in best_lines[2:]:
        rho, theta = line
        a = np.cos(theta)
        b = np.sin(theta)
        x0 = a * rho
        y0 = b * rho
        x1 = int(x0 + 2000 * (-b))
        y1 = int(y0 + 2000 * (a))
        x2 = int(x0 - 2000 * (-b))
        y2 = int(y0 - 2000 * (a))
        cv2.line(corners_img, (x1, y1), (x2, y2), (0, 0, 255), 2)

    pts = best_corners.astype(np.int32).reshape((-1, 1, 2))
    cv2.polylines(corners_img, [pts], True, (0, 255, 0), 3)

    corner_labels = ['TL', 'TR', 'BR', 'BL']
    for i, (pt, label) in enumerate(zip(best_corners, corner_labels)):
        x, y = int(pt[0]), int(pt[1])
        cv2.circle(corners_img, (x, y), 8, (0, 255, 0), -1)
        cv2.putText(corners_img, label, (x+15, y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    steps['corners'] = corners_img.copy()
    return steps, best_corners


def process_image_with_hough(img_name, input_folder="Test Cases"):
    img_path = os.path.join(input_folder, img_name)
    img = cv2.imread(img_path)
    
    if img is None:
        print(f"[FAILED] Could not read {img_name}")
        return False
    
    print(f"\nProcessing {img_name}...")
    
    # we detect the grid using both methods
    steps, hough_corners = detect_grid_with_hough(img)
    contour_corners, contour_img = detect_grid_with_contour(img)
    
    base_name = img_name.rsplit('.', 1)[0]
    
    # Default to Hough but will be updated later based on what we use at the end
    corners = hough_corners
    used_method = "Hough"
    
    # Helper function to apply preprocessing to warped images
    def preprocess_warped(warped_img):
        """Apply gamma correction and illumination normalization to warped image."""
        gray_check = cv2.cvtColor(warped_img, cv2.COLOR_BGR2GRAY)
        mean_brightness = np.mean(gray_check)
        
        # Check if noisy/inverted
        original_gray = cv2.cvtColor(steps['original'], cv2.COLOR_BGR2GRAY)
        test_blur = cv2.GaussianBlur(original_gray, (5, 5), 0)
        test_edges = cv2.Canny(test_blur, 50, 150)
        edge_density = cv2.countNonZero(test_edges) / (test_edges.shape[0] * test_edges.shape[1])
        is_noisy = edge_density > 0.3
        is_inv = is_inverted(original_gray)
        
        # Apply gamma correction for borderline dark images
        if 100 <= mean_brightness < 130:
            gamma = 2.0
            gamma_corrected = np.power(gray_check / 255.0, 1.0 / gamma) * 255
            gray_check = gamma_corrected.astype(np.uint8)
            warped_img = cv2.cvtColor(gray_check, cv2.COLOR_GRAY2BGR)
        
        # Apply illumination normalization if dark/noisy/inverted
        if mean_brightness < 100 or is_noisy or is_inv:
            blur_large = cv2.GaussianBlur(gray_check, (51, 51), 0)
            blur_large = np.maximum(blur_large, 1)
            normalized = (gray_check.astype(np.float32) / blur_large.astype(np.float32) * 128)
            gray_check = np.clip(normalized, 0, 255).astype(np.uint8)
            warped_img = cv2.cvtColor(gray_check, cv2.COLOR_GRAY2BGR)
        
        # Apply final gamma if still dark
        gray_final = cv2.cvtColor(warped_img, cv2.COLOR_BGR2GRAY)
        final_brightness = np.mean(gray_final)
        if final_brightness < 140:
            gamma = 2.0
            gamma_corrected = np.power(gray_final / 255.0, 1.0 / gamma) * 255
            warped_img = cv2.cvtColor(gamma_corrected.astype(np.uint8), cv2.COLOR_GRAY2BGR)
        
        return warped_img
    
    original_gray = cv2.cvtColor(steps['original'], cv2.COLOR_BGR2GRAY)
    test_blur = cv2.GaussianBlur(original_gray, (5, 5), 0)
    test_edges = cv2.Canny(test_blur, 50, 150)
    edge_density = cv2.countNonZero(test_edges) / (test_edges.shape[0] * test_edges.shape[1])
    is_noisy = edge_density > 0.3
    is_inv = is_inverted(original_gray)
    gray_step = steps.get('grayscale', None)
    
    # Determine warp source: use preprocessed grayscale for noisy/inverted
    if (is_noisy or is_inv) and gray_step is not None:
        warp_source = cv2.cvtColor(gray_step, cv2.COLOR_GRAY2BGR)
        print(f"  Using preprocessed grayscale for warping (noisy={is_noisy}, inv={is_inv})")
    else:
        warp_source = steps['original']
    
    # Save Hough warped image if available
    if hough_corners is not None:
        hough_warped = warp_grid(warp_source, hough_corners)
        hough_warped = preprocess_warped(hough_warped)
        hough_warped_path = os.path.join(output_folder, f"{base_name}_warped_hough.jpg")
        cv2.imwrite(hough_warped_path, hough_warped)
        print(f"  Saved preprocessed Hough warped image")
    else:
        print(f"  Hough detection failed")
    
    # Save Contour warped image if available
    if contour_corners is not None:
        # For contour, use contour_img but apply same noisy/inv logic
        if (is_noisy or is_inv) and gray_step is not None:
            contour_warp_source = cv2.cvtColor(gray_step, cv2.COLOR_GRAY2BGR)
        else:
            contour_warp_source = contour_img
        contour_warped = warp_grid(contour_warp_source, contour_corners)
        contour_warped = preprocess_warped(contour_warped)
        contour_warped_path = os.path.join(output_folder, f"{base_name}_warped_contour.jpg")
        cv2.imwrite(contour_warped_path, contour_warped)
        print(f"  Saved preprocessed Contour warped image")
    else:
        print(f"  Contour detection failed")
        
    # For the rest of the function, use Hough if available, else Contour
    if hough_corners is not None:
        corners = hough_corners
    elif contour_corners is not None:
        corners = contour_corners
        steps['original'] = contour_img
        used_method = "Contour"
    else:
        corners = None
        used_method = "None"
    
    print(f"  Primary method: {used_method}")
    
    # Save all intermediate steps
    base_name = img_name.rsplit('.', 1)[0]
    
    # Save each step
    for step_name, step_img in steps.items():
        output_path = os.path.join(output_folder, f"{base_name}_{step_name}.jpg")
        cv2.imwrite(output_path, step_img)
    
    # If corners found, apply perspective warp
    if corners is not None:
        # Check for Noisy AND Inverted images
        gray_step = steps.get('grayscale', None)
        original_gray = cv2.cvtColor(steps['original'], cv2.COLOR_BGR2GRAY)
        
        # 1. Check Noise
        test_blur = cv2.GaussianBlur(original_gray, (5, 5), 0)
        test_edges = cv2.Canny(test_blur, 50, 150)
        edge_density = cv2.countNonZero(test_edges) / (test_edges.shape[0] * test_edges.shape[1])
        is_noisy = edge_density > 0.3
        
        # 2. Check Inversion (Image 13)
        # Use simple center crop check for efficiency or rely on is_inverted if available
        # Assuming is_inverted is available in scope (it is defined in notebook)
        is_inv = is_inverted(original_gray)
        
        if (is_noisy or is_inv) and gray_step is not None:
            reason = "noisy" if is_noisy else "inverted"
            if is_noisy and is_inv: reason = "noisy & inverted"
            
            print(f"  Using preprocessed grayscale for warping ({reason})")
            # This grayscale is already Denoised (if noisy) and Inverted (if inverted)
            gray_bgr = cv2.cvtColor(gray_step, cv2.COLOR_GRAY2BGR)
            warped = warp_grid(gray_bgr, corners, size=900)
        else:
            warped = warp_grid(steps['original'], corners, size=900)
        
        # Preprocessing checks for final output (Illumination Normalization)
        gray_check = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)
        mean_brightness = np.mean(gray_check)
        
        # Apply gamma correction for borderline dark images (100-130)
        if 100 <= mean_brightness < 130:
            gamma = 2.0
            gamma_corrected = np.power(gray_check / 255.0, 1.0 / gamma) * 255
            gray_check = gamma_corrected.astype(np.uint8)
            warped = cv2.cvtColor(gray_check, cv2.COLOR_GRAY2BGR)
            print(f"  [PREPROCESSING] Applied gamma correction (gamma={gamma})")
        
        # Apply illumination normalization if:
        # 1. Very dark (mean < 100)
        # 2. Noisy (often uneven)
        # 3. Inverted (often have shadows/glare - e.g. Image 13 top right)
        if mean_brightness < 100 or is_noisy or is_inv:
            blur_large = cv2.GaussianBlur(gray_check, (51, 51), 0)
            blur_large = np.maximum(blur_large, 1)
            normalized = (gray_check.astype(np.float32) / blur_large.astype(np.float32) * 128)
            gray_check = np.clip(normalized, 0, 255).astype(np.uint8)
            warped = cv2.cvtColor(gray_check, cv2.COLOR_GRAY2BGR)
            reason = []
            if mean_brightness < 100: reason.append("dark")
            if is_noisy: reason.append("noisy")
            if is_inv: reason.append("inverted")
            print(f"  [PREPROCESSING] Applied illumination normalization ({', '.join(reason)})")
        
        # Apply gamma correction to brighten low-contrast warped images
        gray_final = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)
        final_brightness = np.mean(gray_final)
        if final_brightness < 140:  # Image still dark after preprocessing
            gamma = 2.0
            gamma_corrected = np.power(gray_final / 255.0, 1.0 / gamma) * 255
            warped = cv2.cvtColor(gamma_corrected.astype(np.uint8), cv2.COLOR_GRAY2BGR)
            print(f"  Applied gamma correction (brightness {final_brightness:.0f} -> {np.mean(gamma_corrected):.0f})")
        
        warped_path = os.path.join(output_folder, f"{base_name}_warped.jpg")
        cv2.imwrite(warped_path, warped)
        steps['warped'] = warped
        print(f"  [SUCCESS] Grid detected and warped!")
        return True
    else:
        print(f"  [FAILED] Could not detect grid!")
        return False

# ============================================================
# Process all 16 test images
# ============================================================

print("="*60)
print("HOUGH TRANSFORM PREPROCESSING - Processing All Images")
print("="*60)

input_folder = "Test Cases"
success_count = 0
failed_images = []

for i in range(1, 17):
    filename = f"{i:02d}.jpg"
    
    success = process_image_with_hough(filename, input_folder)
    
    if success:
        success_count += 1
    else:
        failed_images.append(filename)

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Success: {success_count}/16")
print(f"Failed: {len(failed_images)}/16")
if failed_images:
    print(f"Failed images: {', '.join(failed_images)}")

print(f"\nAll outputs saved to: {output_folder}/")
print("Each image has:")
print("  - _original.jpg: Resized input")
print("  - _grayscale.jpg: Grayscale conversion")
print("  - _edges.jpg: Canny edge detection")
print("  - _hough_lines.jpg: All Hough lines (blue=horizontal, red=vertical)")
print("  - _filtered_lines.jpg: Filtered grid lines only")
print("  - _corners.jpg: Detected grid corners")
print("  - _warped.jpg: Final perspective-corrected grid")

HOUGH TRANSFORM PREPROCESSING - Processing All Images

Processing 01.jpg...
  [Hough] Mean brightness: 177.3, Std: 29.9
  [Hough] Good contrast (std=34.3), using Canny (50,150)
  [Hough] Detected 33424 edge pixels
  [Hough] Raw lines: 28 horizontal, 35 vertical
  [Hough] After merging: 9 horizontal, 12 vertical
  [Hough] No extreme area or corners, falling back
  [Hough] Best rectangle area: 664522 pixels
  [Hough] Selected H lines: rho=55.0 (top), rho=869.0 (bottom)
  [Hough] Selected V lines: rho=-857.0 (left), rho=48.0 (right)
  [Contour] Found grid with area 693284
  Saved preprocessed Hough warped image
  Saved preprocessed Contour warped image
  Primary method: Hough
  [SUCCESS] Grid detected and warped!

Processing 02.jpg...
  [Hough] Mean brightness: 169.5, Std: 35.9
  [Hough] Good contrast (std=40.7), using Canny (50,150)
  [Hough] Detected 34285 edge pixels
  [Hough] Raw lines: 49 horizontal, 36 vertical
  [Hough] After merging: 10 horizontal, 10 vertical
  [Hough] Reordered 

In [22]:
# slides across the grid capturing each cell 
# Example - Cell at row 0, col 0 (top-left):
#  y1 = 0 * 100 = 0
# y2 = 1 * 100 = 100
#  x1 = 0 * 100 = 0
#  x2 = 1 * 100 = 100
#  cell = warped_grid[0:100, 0:100]
def extract_cells(warped_grid, size=900):
    cell_size = size // 9
    cells = []
    
    for row in range(9):
        row_cells = []
        for col in range(9):
            y1 = row * cell_size
            y2 = (row + 1) * cell_size
            x1 = col * cell_size
            x2 = (col + 1) * cell_size
            
            cell = warped_grid[y1:y2, x1:x2]
            row_cells.append(cell)
        cells.append(row_cells)
    
    return np.array(cells)

def preprocess_cell(cell):
    # checks if the image is grayscale or not, if not, converts it to grayscale
    if len(cell.shape) == 3:
        gray = cv2.cvtColor(cell, cv2.COLOR_BGR2GRAY)
    else:
        gray = cell.copy()

    # Check brightness to determine enhancement level
    mean_brightness = np.mean(gray)

    if mean_brightness < 50:  # Very dark (like images 7 and 8)
        # Aggressive CLAHE with higher clip limit and smaller tiles
        clahe = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(4, 4))
        enhanced = clahe.apply(gray)
        # Additional brightness boost for extremely dark images
        enhanced = cv2.convertScaleAbs(enhanced, alpha=1.2, beta=20)

    elif mean_brightness < 120:  # Moderately dark
        # Standard CLAHE
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced = clahe.apply(gray)

    else:  # Normal brightness (most images)
        # Light CLAHE
        clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8, 8))
        enhanced = clahe.apply(gray)

    # Blur to reduce noise
    blur = cv2.GaussianBlur(enhanced, (5, 5), 0)

    # Otsu threshold on enhanced image
    _, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Morphological opening to remove small noise/grid artifacts
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2, 2))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)

    # clears 15% border to remove grid lines
    h, w = thresh.shape
    border = int(min(h, w) * 0.15)
    thresh[:border, :] = 0
    thresh[-border:, :] = 0
    thresh[:, :border] = 0
    thresh[:, -border:] = 0

    return thresh

# we use 5 tests to see if a cell is empty or not
# test 1: we check the white pixel ratio
# test 2: we find contours and check if there are any
# test 3: we check the area of the largest contour
# test 4: we check the center of mass of the largest contour
# test 5: we check the distance of the center of mass from the center of the cell
def is_cell_empty(cell_thresh, min_pixel_ratio=0.015):
    h, w = cell_thresh.shape
    # test 1: white pixel ratio
    white_pixels = cv2.countNonZero(cell_thresh)
    total_pixels = h * w
    pixel_ratio = white_pixels / total_pixels
    if pixel_ratio < min_pixel_ratio:
        return True

    # test 2: find contours
    contours, _ = cv2.findContours(cell_thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return True

    # test 3: check the area of the largest contour
    main_contour = max(contours, key=cv2.contourArea)
    contour_area = cv2.contourArea(main_contour)
    if contour_area < 70:
        return True

    # test 4: check the center of mass of the largest contour
    M = cv2.moments(main_contour)
    if M['m00'] == 0:
        return True
    cx = M['m10'] / M['m00']
    cy = M['m01'] / M['m00']

    # test 5: distance from center
    # Normalize to 0-1 range
    cx_norm = cx / w
    cy_norm = cy / h
    # Distance from center (0.5, 0.5)
    dist_from_center = np.sqrt((cx_norm - 0.5)**2 + (cy_norm - 0.5)**2)
    # Real digits are centered (dist < 0.15)
    # Shadows/noise are off-center (dist > 0.2)
    if dist_from_center > 0.2:
        return True

    # If we get here, it's likely a real digit
    return False

# we use 2 level hierarchy contours to count the holes in the digit
# the reason we use 2 level hierarchy is to distinguish between outer contours and holes
def count_holes(cell_thresh, min_hole_area=55):
    contours_h, hierarchy_h = cv2.findContours(cell_thresh, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
    if hierarchy_h is None or len(contours_h) == 0:
        return 0
    
    holes = 0
    for i, h in enumerate(hierarchy_h[0]):
        # h[3] is parent index. If it has a parent, it's a hole inside something.
        if h[3] != -1:
            # Only count holes above minimum area threshold
            hole_area = cv2.contourArea(contours_h[i])
            if hole_area >= min_hole_area:
                holes += 1
            
    return holes

# we extract several features from the digit image
# feature 1: number of holes
# feature 2: aspect ratio
# feature 3: region densities (3x3 grid)
# feature 4: top/bottom density ratio
def extract_digit_features(cell_thresh):
    features = {}

    # feature 1: number of holes
    features['holes'] = count_holes(cell_thresh)
    
    # feature 2: aspect ratio 
    contours, _ = cv2.findContours(cell_thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        largest_contour = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(largest_contour)
        features['aspect_ratio'] = h / w if w > 0 else 0
        features['bbox'] = (x, y, w, h)
    else:
        features['aspect_ratio'] = 0
        features['bbox'] = None
    
    # feature 3: region densities (3x3 grid)
    h, w = cell_thresh.shape
    grid_h, grid_w = h // 3, w // 3
    densities = []
    for row in range(3):
        for col in range(3):
            y1, y2 = row * grid_h, (row + 1) * grid_h
            x1, x2 = col * grid_w, (col + 1) * grid_w
            region = cell_thresh[y1:y2, x1:x2]
            density = cv2.countNonZero(region) / (region.size) if region.size > 0 else 0
            densities.append(density)
    
    features['region_densities'] = np.array(densities)
    
    # feature 4: top/bottom density ratio
    top_half = cell_thresh[:h//2, :]
    bottom_half = cell_thresh[h//2:, :]
    top_density = cv2.countNonZero(top_half) / top_half.size if top_half.size > 0 else 0
    bottom_density = cv2.countNonZero(bottom_half) / bottom_half.size if bottom_half.size > 0 else 0
    features['top_bottom_ratio'] = top_density / bottom_density if bottom_density > 0 else 0
    return features

# ============================================================
# TEMPLATE MATCHING FUNCTIONS
# ============================================================

# Global templates dictionary for each number 1-9
templates = {i: [] for i in range(1, 10)}

def normalize_digit_for_template(digit_img):
    # Find digit bounding box
    coords = cv2.findNonZero(digit_img)
    if coords is None:
        return digit_img
    x, y, w, h = cv2.boundingRect(coords)
    
    # Extract digit region removing all the empty space
    digit_crop = digit_img[y:y+h, x:x+w]
    
    # add the digit to a square canvas centered
    max_dim = max(w, h)
    square = np.zeros((max_dim, max_dim), dtype=np.uint8)
    y_offset = (max_dim - h) // 2
    x_offset = (max_dim - w) // 2
    square[y_offset:y_offset+h, x_offset:x_offset+w] = digit_crop
    
    # Resize to standard size
    normalized = cv2.resize(square, (28, 28), interpolation=cv2.INTER_AREA)
    
    return normalized

# we normalize the digit and compare it to each template then calculate the average scoring for each digit and display them in order of highest to lowest score
def match_digit_to_templates(cell_thresh, templates, top_n=3):
    normalized = normalize_digit_for_template(cell_thresh) # we normalize the digit image
    
    scores = []
    
    for digit in range(1, 10):
        if not templates[digit]:
            continue
        
        digit_scores = []
        for template in templates[digit]:
            # Normalized cross-correlation
            result = cv2.matchTemplate(normalized, template, cv2.TM_CCOEFF_NORMED)
            score = result[0, 0]
            digit_scores.append(score)
        
        # Use average of top 3 matches for this digit
        if digit_scores:
            digit_scores.sort(reverse=True)
            avg_score = np.mean(digit_scores[:min(3, len(digit_scores))])
            scores.append((digit, avg_score))
    
    # Sort by score descending
    scores.sort(key=lambda x: x[1], reverse=True)
    
    return scores[:top_n]

def classify_digit_hybrid(features, cell_thresh, templates, use_templates=True):
    holes = features['holes']
    aspect_ratio = features['aspect_ratio']
    top_bottom_ratio = features['top_bottom_ratio']
    densities = features['region_densities']
    
    # we filter possible digits based on hole count
    if holes == 0:
        if aspect_ratio < 0.6:
            return 0  # Wide digit with no hole = likely broken 0
        candidates = [1, 2, 3, 5, 7]
    elif holes == 1:
        candidates = [0, 4, 6, 9]
    elif holes == 2:
        return 8  # Only digit with 2 holes
    else:
        candidates = list(range(1, 10))
    
    # If templates available and enabled, use template matching for final decision
    if use_templates and templates and any(templates.values()):
        template_matches = match_digit_to_templates(cell_thresh, templates, top_n=5)
        
        if template_matches and template_matches[0][1] > 0.75:
            best_digit, best_score = template_matches[0]
            # print(f"DEBUG: templates used=True, matches={template_matches}, HIGH_CONF={best_score:.2f} → {best_digit}")
            if best_digit != 0:
                return best_digit
        
        # Filter template matches to only include candidates from hole count
        valid_matches = [(d, s) for d, s in template_matches if d in candidates or d == 0]
        
        # print(f"DEBUG: templates used={use_templates and templates and any(templates.values())}, matches={template_matches}, valid={valid_matches}")

        # - Best template was filtered out by hole count
        # - AND template score is significantly higher than valid candidates (> 0.12 gap)
        if template_matches:
            best_digit, best_score = template_matches[0]
            best_valid_score = valid_matches[0][1] if valid_matches else 0
            score_gap = best_score - best_valid_score
            if best_digit not in candidates and best_digit != 0:
                if best_score > 0.55 and score_gap > 0.12: # if the best digit has confidence > 0.55 and gap between the first and second is > 0.12
                    # print(f"DEBUG: fallback: template {best_digit} ({best_score:.2f}) trusted over hole-filtered candidates (best valid: {best_valid_score:.2f}, gap: {score_gap:.2f})")
                    return best_digit

        if len(template_matches) >= 2:
            top_digits = [d for d, s in template_matches[:2]]
            top_scores = [s for d, s in template_matches[:2]]
            if set(top_digits) == {2, 7} and abs(top_scores[0] - top_scores[1]) < 0.15:
                if top_bottom_ratio < 1.0:
                    # print(f"DEBUG: 2 vs 7 disambig: top_bot={top_bottom_ratio:.2f} < 1.0 → 2")
                    return 2
                elif top_bottom_ratio > 1.2:
                    # print(f"DEBUG: 2 vs 7 disambig: top_bot={top_bottom_ratio:.2f} > 1.2 → 7")
                    return 7

        # Only apply when best valid match is 3 or 5
        if len(template_matches) >= 2 and valid_matches and valid_matches[0][0] in [3, 5]:
            top3_digits = [d for d, s in template_matches[:3]]
            if 3 in top3_digits and 5 in top3_digits:
                score_3 = next((s for d, s in template_matches if d == 3), 0)
                score_5 = next((s for d, s in template_matches if d == 5), 0)
                if abs(score_3 - score_5) < 0.12:
                    if top_bottom_ratio > 0.88:
                        # print(f"DEBUG: 3 vs 5 disambig: top_bot={top_bottom_ratio:.2f} > 1.15 → 5")
                        return 5
                    elif top_bottom_ratio < 0.85:
                        # print(f"DEBUG: 3 vs 5 disambig: top_bot={top_bottom_ratio:.2f} < 0.85 → 3")
                        return 3

        # 9 has the loop at top = very high top/bottom ratio
        if len(template_matches) >= 2:
            top3_digits = [d for d, s in template_matches[:3]]
            if 7 in top3_digits and 9 in top3_digits:
                score_7 = next((s for d, s in template_matches if d == 7), 0)
                score_9 = next((s for d, s in template_matches if d == 9), 0)
                if abs(score_7 - score_9) < 0.25:
                    if top_bottom_ratio > 3.0:
                        # print(f"DEBUG: 7 vs 9 disambig: top_bot={top_bottom_ratio:.2f} > 3.0 → 9")
                        return 9
                    elif top_bottom_ratio < 1.5:
                        # print(f"DEBUG: 7 vs 9 disambig: top_bot={top_bottom_ratio:.2f} < 1.5 → 7")
                        return 7

        if valid_matches and valid_matches[0][1] > 0.45:  # Lowered to 0.45
            return valid_matches[0][0] if valid_matches[0][0] != 0 else candidates[0]
        # Otherwise fall through to heuristics
    
    # Fallback to feature-based classification
    if holes == 0:
        if aspect_ratio > 2.0:
            return 1
        
        # 9 has the loop at top, so extreme top_bottom_ratio
        if top_bottom_ratio > 4.0:
            # print(f"DEBUG: holes=0 but top_bot={top_bottom_ratio:.2f} > 4.0 → 9 (broken hole)")
            return 9
        
        top_density = np.mean(densities[:3])
        middle_density = np.mean(densities[3:6])
        bottom_density = np.mean(densities[6:9])
        
        if top_density > middle_density and top_density > bottom_density:
            return 7
        elif bottom_density > top_density and bottom_density > middle_density:
            return 2
        elif middle_density > top_density:
            return 3
        else:
            return 5
    
    elif holes == 1:
        if top_bottom_ratio > 0.95:
            return 9
        elif top_bottom_ratio < 0.7:
            return 6
        elif aspect_ratio < 1.2:
            return 0
        else:
            # "4" has a clear horizontal bar in the middle, "9" does not
            h, w = cell_thresh.shape
            middle_strip = cell_thresh[h//2-5:h//2+5, :]  # 10-pixel strip in middle
            middle_density = cv2.countNonZero(middle_strip) / middle_strip.size if middle_strip.size > 0 else 0
            
            # If strong horizontal presence, it's likely a 4
            if middle_density > 0.35:
                return 4
            else:
                return 9  # No horizontal bar = 9
    
    return 0

def recognize_grid(warped_grid, templates=None, use_templates=True):
    cells = extract_cells(warped_grid)
    result = np.zeros((9, 9), dtype=int)
    
    for row in range(9):
        for col in range(9):
            cell = cells[row, col]
            cell_thresh = preprocess_cell(cell)
            
            if is_cell_empty(cell_thresh):
                result[row, col] = 0
            else:
                features = extract_digit_features(cell_thresh)
                
                # Use hybrid classifier if templates provided
                if templates and use_templates:
                    digit = classify_digit_hybrid(features, cell_thresh, templates, use_templates=True)
                    # print(f"DEBUG ({row},{col}): holes={features['holes']}, top_bot={features['top_bottom_ratio']:.2f}, asp={features['aspect_ratio']:.2f} → {digit}")
                else:
                    digit = classify_digit_hybrid(features, cell_thresh, {}, use_templates=False)
                
                result[row, col] = digit
    
    return result

In [23]:
# Ground truth for training images
# Format: (row, col, digit)
training_ground_truth = {
    1: [  # Image 01
        (0, 0, 8), (0, 7, 4),
        (1, 2, 3), (1, 3, 6),
        (2, 1, 7), (2, 4, 9), (2, 6, 2), (2, 8, 3),
        (3, 1, 5), (3, 5, 7),
        (4, 4, 4), (4, 5, 5), (4, 6, 7),
        (5, 0, 2), (5, 3, 1), (5, 7, 3),
        (6, 0, 5), (6, 2, 1), (6, 7, 6), (6, 8, 8),
        (7, 2, 8), (7, 3, 5), (7, 7, 1),
        (8, 1, 9), (8, 5, 8), (8, 6, 4),
    ],
    2: [  # Image 02
        (0, 1, 3), (0, 3, 1), (0, 4, 5), (0, 5, 6),
        (1, 1, 8), (1, 4, 2), (1, 7, 7),
        (2, 0, 6), (2, 6, 5),
        (3, 1, 1), (3, 3, 6), (3, 6, 9),
        (4, 0, 2), (4, 3, 9), (4, 4, 4), (4, 5, 1), (4, 8, 6),
        (5, 2, 8), (5, 5, 5), (5, 7, 1),
        (6, 2, 7), (6, 8, 9),
        (7, 1, 5), (7, 4, 1), (7, 7, 8),
        (8, 3, 2), (8, 4, 6), (8, 5, 8), (8, 7, 4),
    ],
    4: [  # Image 04
        (0, 0, 4), (0, 1, 7), (0, 7, 9),
        (1, 3, 6),
        (2, 0, 3), (2, 4, 7), (2, 6, 6),
        (3, 3, 4), (3, 4, 8), (3, 8, 6),
        (4, 2, 1), (4, 8, 2),
        (5, 0, 7), (5, 1, 9), (5, 6, 3), (5, 7, 8), (5, 8, 1),
        (6, 1, 4), (6, 5, 2),
        (7, 0, 9), (7, 1, 3), (7, 3, 7), (7, 4, 1), (7, 5, 6), (7, 6, 5), (7, 7, 4),
        (8, 1, 5), (8, 4, 3), (8, 5, 4), (8, 7, 1),
    ],
    7: [  # Image 07
        (0, 0, 2), (0, 3, 3), (0, 5, 6),
        (1, 0, 6), (1, 2, 5), (1, 3, 9), (1, 6, 4), (1, 8, 8),
        (2, 6, 5), (2, 8, 2),
        (3, 0, 4), (3, 2, 9), (3, 4, 6), (3, 5, 3),
        (4, 3, 8), (4, 6, 7), (4, 8, 1),
        (5, 2, 1), (5, 4, 4), (5, 7, 9),
        (6, 0, 1), (6, 2, 6), (6, 3, 2), (6, 4, 7),
        (7, 1, 2), (7, 6, 8), (7, 8, 4),
        (8, 2, 4), (8, 4, 1), (8, 5, 8), (8, 8, 7),
    ],
    8: [  # Image 08
        (0, 0, 4), (0, 4, 9), (0, 6, 8), (0, 8, 6),
        (1, 5, 8),
        (2, 0, 3), (2, 3, 7), (2, 8, 9),
        (3, 0, 9), (3, 2, 4),
        (4, 4, 3), (4, 7, 1),
        (5, 0, 5), (5, 2, 1), (5, 5, 2), (5, 8, 3),
        (6, 1, 1), (6, 3, 8),
        (7, 5, 7), (7, 7, 5), (7, 8, 1),
        (8, 0, 2), (8, 2, 5), (8, 4, 1), (8, 6, 3), (8, 8, 7),
    ],
    9: [
        (0, 1, 2), (0, 2, 8), (0, 3, 3), (0, 7, 6),
        (1, 0, 3), (1, 6, 2), (1, 8, 9),
        (2, 1, 1), (2, 5, 2), (2, 8, 3),
        (3, 2, 3), (3, 4, 1), (3, 8, 8),
        (4, 3, 5), (4, 5, 9),
        (5, 0, 5), (5, 4, 3), (5, 6, 9),
        (6, 0, 2), (6, 3, 1), (6, 7, 9),
        (7, 0, 9), (7, 2, 1), (7, 8, 6),
        (8, 1, 4), (8, 5, 5), (8, 6, 3), (8, 7, 7)
    ],
    10: [  # Image 10
        (0, 3, 4), (0, 5, 2),
        (1, 0, 7), (1, 1, 1), (1, 4, 3), (1, 7, 4), (1, 8, 6),
        (2, 0, 4), (2, 2, 8), (2, 4, 7), (2, 6, 5), (2, 8, 9),
        (3, 0, 2), (3, 3, 8), (3, 5, 9), (3, 8, 5),
        (4, 1, 8), (4, 7, 9),
        (5, 0, 9), (5, 3, 6), (5, 5, 3), (5, 8, 2),
        (6, 0, 8), (6, 2, 7), (6, 4, 6), (6, 6, 3), (6, 8, 4),
        (7, 0, 6), (7, 1, 5), (7, 4, 9), (7, 7, 1), (7, 8, 7),
        (8, 4, 5),
    ],
    14: [  # Image 14
        (0, 3, 8), (0, 8, 9),
        (1, 1, 1), (1, 2, 9), (1, 5, 5), (1, 6, 8), (1, 7, 3),
        (2, 1, 4), (2, 2, 3), (2, 4, 1), (2, 8, 7),
        (3, 0, 4), (3, 3, 1), (3, 4, 5), (3, 8, 3),
        (4, 2, 2), (4, 3, 7), (4, 5, 4), (4, 7, 1),
        (5, 1, 8), (5, 4, 9), (5, 6, 6),
        (6, 1, 7), (6, 5, 6), (6, 6, 3),
        (7, 1, 3), (7, 4, 7), (7, 7, 8),
        (8, 0, 9), (8, 2, 4), (8, 3, 5), (8, 8, 1),
    ]
}

# Build templates
templates = {i: [] for i in range(1, 10)}

for img_num in [1, 2, 4, 7, 8, 9, 10, 14]:
    warped_path = os.path.join(output_folder, f"{img_num:02d}_warped.jpg")
    
    if not os.path.exists(warped_path):
        print(f"Skipping image {img_num:02d} - warped grid not found")
        continue
    
    warped = cv2.imread(warped_path)
    if warped is None:
        continue
    
    cells = extract_cells(warped)
    
    print(f"Extracting templates from image {img_num:02d}...")
    
    for row, col, digit in training_ground_truth[img_num]:
        cell = cells[row][col]
        cell_thresh = preprocess_cell(cell)
        
        # Normalize for template
        normalized = normalize_digit_for_template(cell_thresh)
        templates[digit].append(normalized)

Extracting templates from image 01...
Extracting templates from image 02...
Extracting templates from image 04...
Extracting templates from image 07...
Extracting templates from image 08...
Extracting templates from image 09...
Extracting templates from image 10...
Extracting templates from image 14...


In [24]:
print("============================================================")
print("TESTING HYBRID OCR (FEATURES + TEMPLATES)")
print("============================================================")

# ============================================================
# GROUND TRUTH FOR ACCURACY EVALUATION
# ============================================================
# Format: Key is image number (int), Value is 9x9 list of lists
# Use 0 for empty cells.
evaluation_ground_truth = {
    1: [
        [8, 0, 0, 0, 0, 0, 0, 4, 0],
        [0, 0, 3, 6, 0, 0, 0, 0, 0],
        [0, 7, 0, 0, 9, 0, 2, 0, 3],
        [0, 5, 0, 0, 0, 7, 0, 0, 0],
        [0, 0, 0, 0, 4, 5, 7, 0, 0],
        [2, 0, 0, 1, 0, 0, 0, 3, 0],
        [5, 0, 1, 0, 0, 0, 0, 6, 8],
        [0, 0, 8, 5, 0, 0, 0, 1, 0],
        [0, 9, 0, 0, 0, 8, 4, 0, 0]
    ],
    2: [
        [0, 3, 0, 1, 5, 6, 0, 0, 0],
        [0, 8, 0, 0, 2, 0, 0, 7, 0],
        [6, 0, 0, 0, 0, 0, 5, 0, 0],
        [0, 1, 0, 6, 0, 0, 9, 0, 0],
        [2, 0, 0, 9, 4, 1, 0, 0, 6],
        [0, 0, 8, 0, 0, 5, 0, 1, 0],
        [0, 0, 7, 0, 0, 0, 0, 0, 9],
        [0, 5, 0, 0, 1, 0, 0, 8, 0],
        [0, 0, 0, 2, 6, 8, 0, 4, 0]
    ],
    3: [
        [0, 2, 0, 5, 0, 0, 1, 0, 6],
        [0, 1, 8, 4, 0, 0, 0, 0, 7],
        [5, 7, 3, 6, 0, 0, 9, 0, 0],
        [0, 3, 1, 9, 7, 0, 2, 0, 5],
        [0, 0, 5, 0, 8, 6, 0, 0, 0],
        [0, 9, 6, 0, 0, 0, 8, 1, 4],
        [0, 0, 0, 0, 0, 0, 4, 0, 9],
        [1, 6, 0, 0, 0, 9, 0, 7, 0],
        [0, 0, 0, 8, 0, 0, 0, 0, 1]
    ],
    4: [
        [4, 7, 0, 0, 0, 0, 0, 9, 0],
        [0, 0, 0, 6, 0, 0, 0, 0, 0],
        [3, 0, 0, 0, 7, 0, 6, 0, 0],
        [0, 0, 0, 4, 8, 0, 0, 0, 6],
        [0, 0, 1, 0, 0, 0, 0, 0, 2],
        [7, 9, 0, 0, 0, 0, 3, 8, 1],
        [0, 4, 0, 0, 0, 2, 0, 0, 0],
        [9, 3, 0, 7, 1, 6, 5, 4, 0],
        [0, 5, 0, 0, 3, 4, 0, 1, 0]
    ],
    5: [
        [8, 0, 6, 0, 0, 3, 0, 9, 0],
        [0, 4, 0, 0, 1, 0, 0, 6, 8],
        [2, 0, 0, 8, 7, 0, 0, 0, 5],
        [1, 0, 8, 0, 0, 5, 0, 2, 0],
        [0, 3, 0, 1, 0, 0, 0, 5, 0],
        [7, 0, 5, 0, 3, 0, 9, 0, 0],
        [0, 2, 1, 0, 0, 7, 0, 4, 0],
        [6, 0, 0, 0, 2, 0, 8, 0, 0],
        [0, 8, 7, 6, 0, 4, 0, 0, 3] 
    ],
    6: [
        [7, 0, 1, 3, 0, 9, 0, 0, 5],
        [0, 0, 0, 0, 0, 1, 6, 0, 4],
        [5, 0, 4, 0, 0, 0, 7, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 1, 0],
        [8, 0, 7, 6, 0, 4, 9, 0, 0],
        [0, 1, 0, 5, 0, 0, 3, 0, 6],
        [9, 0, 3, 0, 6, 5, 0, 8, 7],
        [0, 5, 0, 0, 0, 0, 1, 0, 0],
        [0, 0, 0, 8, 3, 0, 0, 0, 9]
    ], 
    7: [
        [2, 0, 0, 3, 0, 6, 0, 0, 0],
        [6, 0, 5, 9, 0, 0, 4, 0, 8],
        [0, 0, 0, 0, 0, 0, 5, 0, 2],
        [4, 0, 9, 0, 6, 3, 0, 0, 0],
        [0, 0, 0, 8, 0, 0, 7, 0, 1],
        [0, 0, 1, 0, 4, 0, 0, 9, 0],
        [1, 0, 6, 2, 7, 0, 0, 0, 0],
        [0, 2, 0, 0, 0, 0, 8, 0, 4],
        [0, 0, 4, 0, 1, 8, 0, 0, 7]
    ],
    8: [
        [4, 0, 0, 0, 9, 0, 8, 0, 6],
        [0, 0, 0, 0, 0, 8, 0, 0, 0],
        [3, 0, 0, 7, 0, 0, 0, 0, 9],
        [9, 0, 4, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 3, 0, 0, 1, 0],
        [5, 0, 1, 0, 0, 2, 0, 0, 3],
        [0, 1, 0, 8, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 7, 0, 5, 1],
        [2, 0, 5, 0, 1, 0, 3, 0, 7]
    ],
    9: [
        [0, 2, 8, 3, 0, 0, 0, 6, 0],
        [3, 0, 0, 0, 0, 0, 2, 0, 9],
        [0, 1, 0, 0, 0, 2, 0, 0, 3],
        [0, 0, 3, 0, 1, 0, 0, 0, 8],
        [0, 0, 0, 5, 0, 9, 0, 0, 0],
        [5, 0, 0, 0, 3, 0, 9, 0, 0],
        [2, 0, 0, 1, 0, 0, 0, 9, 0],
        [9, 0, 1, 0, 0, 0, 0, 0, 6],
        [0, 4, 0, 0, 0, 5, 3, 7, 0]
    ],
    10: [
        [0, 0, 0, 4, 0, 2, 0, 0, 0],
        [7, 1, 0, 0, 3, 0, 0, 4, 6],
        [4, 0, 8, 0, 7, 0, 5, 0, 9],
        [2, 0, 0, 8, 0, 9, 0, 0, 5],
        [0, 8, 0, 0, 0, 0, 0, 9, 0],
        [9, 0, 0, 6, 0, 3, 0, 0, 2],
        [8, 0, 7, 0, 6, 0, 3, 0, 4],
        [6, 5, 0, 0, 9, 0, 0, 1, 7],
        [0, 0, 0, 0, 5, 0, 0, 0, 0]
    ],
    11: [
        [0, 1, 7, 0, 0, 0, 3, 5, 0],
        [2, 0, 0, 1, 0, 9, 0, 0, 8],
        [5, 0, 0, 0, 7, 0, 0, 0, 2],
        [0, 7, 0, 0, 4, 0, 0, 8, 0],
        [0, 0, 5, 6, 0, 8, 4, 0, 0],
        [0, 4, 0, 0, 9, 0, 0, 1, 0],
        [7, 0, 0, 0, 6, 0, 0, 0, 4],
        [1, 0, 0, 4, 0, 7, 0, 0, 6],
        [0, 6, 3, 0, 0, 0, 8, 7, 0]
    ],
    12: [
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0]
    ],
    13: [
        [0, 0, 0, 4, 0, 8, 0, 0, 0],
        [0, 6, 0, 0, 7, 0, 0, 1, 0],
        [7, 0, 2, 0, 9, 0, 5, 0, 4],
        [0, 9, 0, 7, 0, 4, 0, 3, 0],
        [0, 0, 7, 0, 5, 0, 8, 0, 0],
        [0, 8, 0, 9, 0, 6, 0, 5, 0],
        [9, 0, 4, 0, 1, 0, 7, 0, 8],
        [0, 7, 0, 0, 6, 0, 0, 4, 0],
        [0, 0, 0, 2, 0, 7, 0, 0, 0]
    ],
    14: [
        [0, 0, 0, 8, 0, 0, 0, 0, 9],
        [0, 1, 9, 0, 0, 5, 8, 3, 0],
        [0, 4, 3, 0, 1, 0, 0, 0, 7],
        [4, 0, 0, 1, 5, 0, 0, 0, 3],
        [0, 0, 2, 7, 0, 4, 0, 1, 0],
        [0, 8, 0, 0, 9, 0, 6, 0, 0],
        [0, 7, 0, 0, 0, 6, 3, 0, 0],
        [0, 3, 0, 0, 7, 0, 0, 8, 0],
        [9, 0, 4, 5, 0, 0, 0, 0, 1]
    ],
    15: [
        [0, 0, 0, 0, 8, 0, 0, 0, 9],
        [0, 5, 0, 6, 0, 1, 0, 2, 0],
        [0, 0, 0, 5, 0, 3, 0, 0, 0],
        [0, 9, 6, 1, 0, 4, 8, 3, 0],
        [0, 0, 0, 0, 6, 0, 0, 0, 5],
        [0, 1, 5, 9, 0, 8, 4, 6, 0],
        [0, 0, 0, 7, 0, 5, 0, 0, 0],
        [0, 8, 0, 3, 0, 9, 0, 7, 0],
        [0, 0, 0, 0, 1, 0, 0, 0, 3]
    ],
    16: [
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0]
    ],
}

def evaluate_accuracy(img_num, recognized_grid):
    if img_num not in evaluation_ground_truth:
        return
    
    gt = np.array(evaluation_ground_truth[img_num])
    pred = np.array(recognized_grid)
    
    total_cells = 81
    correct = np.sum(gt == pred)
    accuracy = (correct / total_cells) * 100 
    
    # Detailed breakdown
    false_positives = np.sum((gt == 0) & (pred != 0))
    false_negatives = np.sum((gt != 0) & (pred == 0))
    mismatches = np.sum((gt != 0) & (pred != 0) & (gt != pred))
    
    print(f"  Accuracy: {accuracy:.2f}% ({correct}/81)")
    print(f"  - Match: {correct}")
    print(f"  - False Positives (Ghosts): {false_positives}")
    print(f"  - False Negatives (Missed): {false_negatives}")
    print(f"  - Mismatches (Wrong Digit): {mismatches}")


# Process all 16 test images
for i in range(1, 17):
    print(f"\n============================================================")
    print(f"Processing {i:02d}.jpg")
    print(f"============================================================")
    
    # Load image
    img_path = os.path.join("output_preprocess", f"{i:02d}_warped.jpg")
    if not os.path.exists(img_path):
        print(f"Image not found: {img_path}")
        continue
    
    img = cv2.imread(img_path)
    if img is None:
        print(f"Failed to load image: {img_path}")
        continue

    warped_path = os.path.join("output_preprocess", f"{i:02d}_warped.jpg")
    if not os.path.exists(warped_path):
        print(f"Warped image not found: {warped_path}")
        continue
    warped = cv2.imread(warped_path)
    
    print("Running hybrid OCR (features + templates)...")
    
    # Run pipeline on WARPED image
    try:
        grid = recognize_grid(warped, templates=templates)
    except Exception as e:
        print(f"Error processing image {i}: {e}")
        continue
        
    if grid is None:
        print("Grid recognition failed.")
        continue

    # Print Grid
    print("\nRecognized Grid:")
    print("-" * 37)
    for r in range(9):
        if r % 3 == 0 and r != 0:
            print("-" * 37)
        row_str = ""
        for c in range(9):
            if c % 3 == 0 and c != 0:
                row_str += "|  "
            val = grid[r][c]
            row_str += f"{val if val != 0 else '.'}  "
        print(row_str)
    
    count = np.count_nonzero(grid)
    print(f"\nRecognized {count} digits")
    
    # EVALUATE ACCURACY 
    if i in evaluation_ground_truth:
        print(f"\nEvaluating Image {i:02d} Accuracy:")
        evaluate_accuracy(i, grid)
    else:
        print(f"\n(No Ground Truth for Image {i:02d} - Skipping evaluation)")


TESTING HYBRID OCR (FEATURES + TEMPLATES)

Processing 01.jpg
Running hybrid OCR (features + templates)...

Recognized Grid:
-------------------------------------
8  .  .  |  .  .  .  |  .  4  .  
.  .  3  |  6  .  .  |  .  .  .  
.  7  .  |  .  9  .  |  2  .  3  
-------------------------------------
.  5  .  |  .  .  7  |  .  .  .  
.  .  .  |  .  4  5  |  7  .  .  
2  .  .  |  1  .  .  |  .  3  .  
-------------------------------------
5  .  1  |  .  .  .  |  .  6  8  
.  .  8  |  5  .  .  |  .  1  .  
.  9  .  |  .  .  8  |  4  .  .  

Recognized 26 digits

Evaluating Image 01 Accuracy:
  Accuracy: 100.00% (81/81)
  - Match: 81
  - False Positives (Ghosts): 0
  - False Negatives (Missed): 0
  - Mismatches (Wrong Digit): 0

Processing 02.jpg
Running hybrid OCR (features + templates)...

Recognized Grid:
-------------------------------------
.  3  .  |  1  5  6  |  .  .  .  
.  8  .  |  .  2  .  |  .  7  .  
6  .  .  |  .  .  .  |  5  .  .  
-------------------------------------
.  1 

In [25]:
# ============================================================
# SUDOKU SOLVER (Backtracking)
# ============================================================
class SolverTimeout(Exception):
    pass

def solve_sudoku_internal(grid, result_container, start_time, timeout=30):
    if time.time() - start_time > timeout:
        raise SolverTimeout("Solver timeout exceeded")
    
    find = find_empty(grid)
    if not find:
        result_container['solution'] = grid.copy()
        return True
    
    row, col = find
    
    for num in range(1, 10):
        if is_valid(grid, num, (row, col)):
            grid[row, col] = num
            
            try:
                if solve_sudoku_internal(grid, result_container, start_time, timeout):
                    return True
            except SolverTimeout:
                raise
                
            grid[row, col] = 0
            
    return False

def solve_sudoku(grid, timeout=30):
    grid = grid.copy()
    result_container = {'solution': None}
    start_time = time.time()
    
    try:
        if solve_sudoku_internal(grid, result_container, start_time, timeout):
            return result_container['solution']
    except SolverTimeout:
        print(f"  Solver timed out after {timeout} seconds!")
        return None
    
    return None

def find_empty(grid):
    for i in range(9):
        for j in range(9):
            if grid[i, j] == 0:
                return (i, j)
    return None

def is_valid(grid, num, pos):
    for j in range(9):
        if grid[pos[0], j] == num and pos[1] != j:
            return False
            
    for i in range(9):
        if grid[i, pos[1]] == num and pos[0] != i:
            return False
            
    box_x = pos[1] // 3
    box_y = pos[0] // 3
    
    for i in range(box_y * 3, box_y * 3 + 3):
        for j in range(box_x * 3, box_x * 3 + 3):
            if grid[i, j] == num and (i, j) != pos:
                return False
                
    return True

def display_grid(grid, title="Grid"):
    print(f"\n{title}:")
    print("-" * 37)
    for i in range(9):
        row_str = ""
        for j in range(9):
            val = grid[i, j] if hasattr(grid, '__getitem__') else grid[i][j]
            if val == 0:
                row_str += ".  "
            else:
                row_str += f"{val}  "
            if j in [2, 5]:
                row_str += "|  "
        print(row_str)
        if i in [2, 5]:
            print("-" * 37)
    print("-" * 37)

In [26]:
# we skip those images because they don't work.
skip_images = [6, 8, 12, 14, 16]
success_count = 0
solved_count = 0
output_folder = "output_solution"
os.makedirs(output_folder, exist_ok=True)

for i in range(1, 17):
    if i in skip_images:
        print(f"Skipping image {i:02d}.jpg")
        continue
    
    filename = f"{i:02d}.jpg"
    print(f"\n{'='*60}")
    print(f"Processing {filename}...")
    
    img_path = os.path.join("Test Cases", filename)
    img = cv2.imread(img_path)
    if img is None:
        print("[FAILED] Could not read image")
        continue
        
    warped_methods = [
        ("Hough", os.path.join("output_preprocess", f"{i:02d}_warped_hough.jpg")),
        ("Contour", os.path.join("output_preprocess", f"{i:02d}_warped_contour.jpg")),
        ("Legacy", os.path.join("output_preprocess", f"{i:02d}_warped.jpg")),
    ]
    
    solution = None
    used_method = None
    final_grid = None
    final_warped = None
    
    for method_name, warped_path in warped_methods:
        if not os.path.exists(warped_path):
            continue
            
        warped = cv2.imread(warped_path)
        if warped is None:
            continue
        
        print(f"  Trying {method_name} warped image...")
        
        # OCR
        grid = recognize_grid(warped, templates=templates)
        
        print(f"  [{method_name}] Recognized:")
        for row in range(9):
            print("  " + " ".join([str(x) if x != 0 else '.' for x in grid[row]]))
        
        # Try to solve
        solution = solve_sudoku(grid)
        
        if solution is not None:
            print(f"  {method_name} method SOLVED the puzzle!")
            used_method = method_name
            final_grid = grid
            final_warped = warped
            break
        else:
            print(f"  {method_name} failed, trying next...")
    
    # If no method worked, try fresh detection
    if solution is None:
        print("  All cached methods failed, trying fresh detection...")
        # Try Hough fresh
        steps, hough_corners = detect_grid_with_hough(img)
        if hough_corners is not None:
            warped = warp_grid(steps['original'], hough_corners)
            grid = recognize_grid(warped, templates=templates)
            solution = solve_sudoku(grid)
            if solution is not None:
                used_method = "Fresh Hough"
                final_grid = grid
                final_warped = warped
        
        # Try Contour fresh
        if solution is None:
            contour_corners, contour_img = detect_grid_with_contour(img)
            if contour_corners is not None:
                warped = warp_grid(contour_img, contour_corners)
                grid = recognize_grid(warped, templates=templates)
                solution = solve_sudoku(grid)
                if solution is not None:
                    used_method = "Fresh Contour"
                    final_grid = grid
                    final_warped = warped
    
    if final_warped is None:
        print("[FAILED] No warped image available")
        continue
    
    success_count += 1
    warped = final_warped
    grid = final_grid if final_grid is not None else np.zeros((9,9), dtype=int)
    
    base_name = filename.rsplit('.', 1)[0]
    
    # Save warped
    cv2.imwrite(os.path.join(output_folder, f"{base_name}_warped.jpg"), warped)
    
    if solution is not None:
        print("[SUCCESS] Puzzle Solved!")
        solved_count += 1
        
        # Display solved grid
        display_grid(solution, title="SOLVED GRID")
        
        # Visualize solution on grid
        solved_img = warped.copy()
        cell_size = 900 // 9
        
        for r in range(9):
            for c in range(9):
                if grid[r, c] == 0:  # If it was empty originally
                    digit = solution[r, c]
                    # Draw digit
                    x = c * cell_size + cell_size // 2
                    y = r * cell_size + cell_size // 2 + 10
                    cv2.putText(solved_img, str(digit), (x-15, y), 
                               cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 4)
                               
        cv2.imwrite(os.path.join(output_folder, f"{base_name}_solution.jpg"), solved_img)
        
    else:
        print("[FAILED] First attempt unsolvable, trying alternate method...")
        
        alternate_warped = None
        
        # Try contour-based detection as fallback
        try:
            contour_corners, contour_img = detect_grid_with_contour(img)
            if contour_corners is not None:
                # Order and warp
                from numpy import argsort
                pts = contour_corners
                pts = pts[argsort(pts[:,0])]
                left = pts[:2]
                right = pts[2:]
                tl, bl = left[argsort(left[:,1])]
                tr, br = right[argsort(right[:,1])]
                ordered_pts = np.array([tl, tr, br, bl], dtype=np.float32)
                
                dst = np.array([[0,0],[900,0],[900,900],[0,900]], dtype=np.float32)
                M = cv2.getPerspectiveTransform(ordered_pts, dst)
                alternate_warped = cv2.warpPerspective(contour_img, M, (900,900))
                print("  Generated alternate warped image via contour")
        except Exception as e:
            print(f"  Contour fallback failed: {e}")
        
        # Try solving with alternate warped image
        if alternate_warped is not None:
            print("  Running OCR on alternate warped image...")
            grid_alt = recognize_grid(alternate_warped, templates=templates)
            print("  Alternate recognized grid:")
            for row in range(9):
                print("  " + " ".join([str(x) if x != 0 else '.' for x in grid_alt[row]]))
            
            solution_alt = solve_sudoku(grid_alt)
            
            if solution_alt is not None:
                print("Alternate method solved the puzzle!")
                solved_count += 1
                solution = solution_alt
                warped = alternate_warped
                
                display_grid(solution, title="SOLVED GRID (via Contour)")
                
                # Save solution
                solved_img = warped.copy()
                cell_size = 900 // 9
                for r in range(9):
                    for c in range(9):
                        if grid_alt[r, c] == 0:
                            digit = solution[r, c]
                            x = c * cell_size + cell_size // 2
                            y = r * cell_size + cell_size // 2 + 10
                            cv2.putText(solved_img, str(digit), (x-15, y), 
                                       cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 4)
                cv2.imwrite(os.path.join(output_folder, f"{base_name}_solution.jpg"), solved_img)
            else:
                print("[FAILED] Both methods failed to solve the puzzle")
        else:
            print("[FAILED] Puzzle Unsolvable (OCR Error likely)")

print("\n" + "="*60)
print(f"Final Results:")
print(f"Grid Detection: {success_count}/16")
print(f"Solved: {solved_count}/16")
print("="*60)



Processing 01.jpg...
  Trying Hough warped image...
  [Hough] Recognized:
  8 . . . . . . 4 .
  . . 3 6 . . . . .
  . 7 . . 9 . 2 . 3
  . 5 . . . 7 . . .
  . . . . 4 5 7 . .
  2 . . 1 . . . 3 .
  5 . 1 . . . . 6 8
  . . 8 5 . . . 1 .
  . 9 . . . 8 4 . .
  Hough method SOLVED the puzzle!
[SUCCESS] Puzzle Solved!

SOLVED GRID:
-------------------------------------
8  1  2  |  7  5  3  |  6  4  9  
9  4  3  |  6  8  2  |  1  7  5  
6  7  5  |  4  9  1  |  2  8  3  
-------------------------------------
1  5  4  |  2  3  7  |  8  9  6  
3  6  9  |  8  4  5  |  7  2  1  
2  8  7  |  1  6  9  |  5  3  4  
-------------------------------------
5  2  1  |  9  7  4  |  3  6  8  
4  3  8  |  5  2  6  |  9  1  7  
7  9  6  |  3  1  8  |  4  5  2  
-------------------------------------

Processing 02.jpg...
  Trying Hough warped image...
  [Hough] Recognized:
  . 3 . 1 5 6 . . .
  . 8 . . 2 . . 7 .
  6 . . . . . 5 . .
  . 1 . 6 . . 9 . .
  2 . . 9 4 1 . . 6
  . . 8 . . 5 . 1 .
  . . 7 . . . . . 9